# Vẽ Ảnh Bàn Cờ Tùy Chỉnh & Toàn Bộ Nước Đi Của Agent Level 1

Notebook này minh họa cách tái sử dụng `render_board_image()` (`chess_ai/ui/board_renderer.py`) để vẽ một thế cờ bất kỳ từ chuỗi `FEN` và hiển thị **tất cả các nước đi hợp lệ** mà `Level1RandomAgent` có thể chọn (xác suất đồng đều $1/N$), kèm nước đi được bốc ngẫu nhiên thực tế.

- **Mũi tên xanh dương:** Tất cả các nước hợp lệ mà Agent Level 1 có thể chọn.
- **Mũi tên xanh lá:** Nước đi được `Level1RandomAgent.select_move()` chọn ngẫu nhiên ở lần chạy hiện tại.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from chess_ai.agents.level1_random import Level1RandomAgent
from chess_ai.core.chess_env import ChessEnv
from chess_ai.ui.board_renderer import render_board_image

In [ ]:
# Bạn có thể thay bất kỳ chuỗi FEN nào vào đây
CUSTOM_FEN = "4k3/2p2p2/1n6/4P3/2B2N2/8/4K3/3R4 w - - 0 1"
SEED = 42

env = ChessEnv(fen=CUSTOM_FEN)
agent = Level1RandomAgent(seed=SEED)

legal_moves = env.legal_actions()
chosen_move = agent.select_move(env)
prob = 1.0 / len(legal_moves)

print(f"FEN: {CUSTOM_FEN}")
print(f"Lượt đi: {'Trắng' if env.current_player() == 1 else 'Đen'}")
print(f"Tổng số nước Agent Level 1 có thể chọn: {len(legal_moves)} (xác suất mỗi nước = {prob:.4f})")
print(f"Nước Agent Level 1 chọn ngẫu nhiên (seed={SEED}): {chosen_move.uci()} ({env.board.san(chosen_move)})")

img = render_board_image(
    env.board,
    move=chosen_move,
    moves=legal_moves,
    board_size=512,
)

fig, ax = plt.subplots(figsize=(6.2, 6.2), dpi=130)
ax.imshow(img)
ax.set_title(
    f"Agent Level 1 — {len(legal_moves)} nước hợp lệ (xanh dương) | Chọn: {env.board.san(chosen_move)} (xanh lá)",
    fontsize=10,
)
ax.axis("off")
fig.tight_layout()
plt.show()

# Liệt kê đầy đủ các nước đi hợp lệ mà Level 1 có thể chọn
df_moves = pd.DataFrame([
    {
        "STT": idx + 1,
        "UCI": m.uci(),
        "SAN": env.board.san(m),
        "Xác suất": round(prob, 4),
        "Được chọn": "✓" if m == chosen_move else "",
    }
    for idx, m in enumerate(legal_moves)
])
display(df_moves)